In [ ]:
import os
from pathlib import Path
import pandas as pd
from datetime import datetime
import chardet

In [ ]:
def strip_new_line(line: str) -> str:
    return line.strip("\n")

In [ ]:
def convert_str_to_datetime(datetime_str: str) -> datetime:
    datetime_obj = datetime.strptime(datetime_str, '%Y-%m-%d %H:%M:%S.%f')
    return datetime_obj

In [ ]:
def read_dat_to_df_with_time_stamp(file_path: Path) -> pd.DataFrame:
    # Step 1: Detect encoding
    with open(file_path, 'rb') as f:
        raw_data = f.read(10000)
        encoding = chardet.detect(raw_data)['encoding'] or 'utf-8'

    # Step 2: Read lines, skip header
    with open(file_path, 'r', encoding=encoding) as f:
        lines = f.readlines()[5:]  # skip first 5 metadata lines

    # Step 3: Parse data lines
    records = []
    for line in lines:
        parts = line.strip().split(',')
        if len(parts) >= 2:
            try:
                value = float(parts[0].strip())
                timestamp_str = parts[1].strip()
                timestamp = pd.to_datetime(timestamp_str, errors='coerce')
                if pd.notnull(timestamp):
                    records.append((timestamp, value))
                else:
                    print(f"[DEBUG] Invalid timestamp in line: {line.strip()}")
            except ValueError:
                continue  # Skip lines that can't be parsed
        

    # Step 4: Create DataFrame
    if not records:
        print(f"[WARNING] No valid timestamped data found in: {file_path.name}")
        return pd.DataFrame()
    

    df = pd.DataFrame(records, columns=['timestamp', 'value'])
    df = df.sort_values(by='timestamp').reset_index(drop=True)

    return df


In [ ]:
def read_sensor_dat(sensor_paths: list[Path]) -> pd.DataFrame:
    print(f"[DEBUG] Reading sensor data from {len(sensor_paths)} files: {[str(p.name) for p in sensor_paths]}")

    sensor_df = pd.DataFrame()

    # Filter out any "._" files (macOS metadata files)
    clean_paths = [p for p in sensor_paths if not p.name.startswith("._")]

    for dat_file in clean_paths:
        dat_df = read_dat_to_df_with_time_stamp(file_path=dat_file)
        
        if dat_df.empty:
            print(f"[WARNING] No valid data in file: {dat_file}")
            continue
        
        sensor_df = pd.concat([sensor_df, dat_df], sort=True)
    
    if 'timestamp' not in sensor_df.columns or sensor_df.empty:
        print(f"[ERROR] No timestamp column found after reading files: {[str(p) for p in clean_paths]}")
        return pd.DataFrame()
    
    sensor_df = sensor_df.sort_values(by="timestamp", inplace=False)
    return sensor_df


In [ ]:
def find_synchronized_start_end_time(day_dict: dict[str, pd.DataFrame]):
    start_times = [df.iloc[0]["timestamp"] for df in day_dict.values()]
    end_times = [df.iloc[-1]["timestamp"] for df in day_dict.values()]
    synchronized_start_time = max(start_times)
    synchronized_end_time = min(end_times)
    return synchronized_start_time, synchronized_end_time

In [ ]:
def trim_data(day_dict: dict) -> dict:
    synchronized_start_time, synchronized_end_time = find_synchronized_start_end_time(day_dict)
    return {sensor: df.loc[(df["timestamp"] >= synchronized_start_time) &
                           (df["timestamp"] <= synchronized_end_time)]
            for sensor, df in day_dict.items()}

In [ ]:
def preprocess_inlab_sensor_data(subject_rings_path: Path, output_dir: Path):
    sensor_types = ["left_finger", "left_wrist", "right_finger", "right_wrist"]
    sensor_paths = {stype: list(subject_rings_path.glob(f'*{stype}*.dat')) for stype in sensor_types}
    
    sensor_data = {}
    for stype, paths in sensor_paths.items():
        print(f"[INFO] Found {len(paths)} file(s) for sensor type: {stype}")
        df = read_sensor_dat(paths)
        if not df.empty:
            print(f"[INFO] Successfully read data for {stype}: {len(df)} records")
            sensor_data[stype.replace('_', ' ')] = df
        else:
            print(f"[INFO] Skipping {stype} — no valid data.")

    if not sensor_data:
        print(f"[WARNING] No sensor data found for: {subject_rings_path}")
        return  # Skip this subject

    trimmed_data = trim_data(sensor_data)

    subject = subject_rings_path.parents[2].name  # subject ID folder name
    output_subdir = output_dir / subject 
    output_subdir.mkdir(parents=True, exist_ok=True)

    for sensor_side, df in trimmed_data.items():
        output_file = output_subdir / f"{sensor_side}.csv"
        df.to_csv(output_file, index=False)
        print(f"[SUCCESS] Saved {sensor_side} data to {output_file} ({len(df)} records)")


In [ ]:
def process_discharge_inlab_data(data_path: Path, output_dir: Path):
    subjects = [d for d in os.listdir(data_path) if not d.startswith('.')]
    for subject in subjects:
        rings_path = data_path / subject / "Discharge" / "InLab" / "Rings"
        if rings_path.exists():
            preprocess_inlab_sensor_data(rings_path, output_dir)

In [ ]:
process_discharge_inlab_data(
    data_path=Path("path/to/your/data"),
    output_dir=Path("path/to/your/data")
)